In [1]:
# installs the Requests Python library so your code can talk to the internet—cleanly, reliably, and without pain.

# !pip install requests
import requests
import xml.etree.ElementTree as ET

print(requests.__version__)


2.32.4


In [2]:
# Folder Setup in Colab
!mkdir -p pubmed_rag/tools

In [3]:
import os
os.environ["E-utils_NCBI"] = "my_secret_key"

In [4]:
NCBI_API_KEY = os.getenv("E-utils_NCBI")
print("API Key loaded:", NCBI_API_KEY is not None)

API Key loaded: True


# ==============================
# PubMed Ingestion (API Key Enabled)

In [5]:
# ==============================
# PubMed Ingestion (Improved for RAG)
# ==============================
import json
import requests
import xml.etree.ElementTree as ET
import time
import os
import re

NCBI_API_KEY = os.environ.get("E-utils_NCBI")  # optional


# ----------- SEARCH -----------
def search_pubmed(query: str, retmax: int = 50):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax
    }

    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    return r.json().get("esearchresult", {}).get("idlist", [])


# ----------- FETCH ------------
def fetch_pubmed_article(pmid: str, max_retries: int = 3):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

    params = {
        "db": "pubmed",
        "id": pmid,
        "retmode": "xml"
    }

    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY

    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=30)

        if r.status_code == 429:
            wait = 2 ** attempt
            print(f"⚠️ Rate limited for PMID {pmid}. Waiting {wait}s...")
            time.sleep(wait)
            continue

        r.raise_for_status()
        break
    else:
        print(f"❌ Failed to fetch PMID {pmid}")
        return None

    root = ET.fromstring(r.text)

    title = root.findtext(".//ArticleTitle", default="").strip()
    journal = root.findtext(".//Journal/Title", default="").strip()

    abstract_parts = [
        elem.text.strip()
        for elem in root.findall(".//AbstractText")
        if elem.text
    ]
    abstract = " ".join(abstract_parts)

    year = root.findtext(".//PubDate/Year")
    if not year:
        medline_date = root.findtext(".//PubDate/MedlineDate")
        year = medline_date.split(" ")[0] if medline_date else ""

    return {
        "pmid": pmid,
        "title": title,
        "journal": journal,
        "year": year,
        "abstract": abstract,
        "text": title + ". " + abstract
    }


# ----------- CLEAN ABSTRACTS -----------
def clean_abstract(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r"\s+", " ", text)

    section_headers = [
        r"\bbackground\b:",
        r"\bobjective\b:",
        r"\bobjectives\b:",
        r"\bmethods\b:",
        r"\bmethodology\b:",
        r"\bresults\b:",
        r"\bconclusion\b:",
        r"\bconclusions\b:",
        r"\bsignificance\b:"
    ]
    for pattern in section_headers:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)

    text = re.sub(r"\s+([.,;:])", r"\1", text)

    return text.strip()


# ----------- RUN ------------
print("🔬 PubMed Search Tool")
print("-" * 40)

QUERY = input("Enter PubMed search query: ")

safe_query = re.sub(r'[^a-zA-Z0-9]+', '_', QUERY).strip('_')
OUTPUT_FILE = f"{safe_query}_pubmed.json"
CLEAN_FILE = f"{safe_query}_pubmed_clean.json"

print("\n🔍 Searching PubMed...")
pmids = search_pubmed(QUERY, retmax=50)
print("Total PMIDs found:", len(pmids))

articles = []
cleaned_records = []

for pmid in pmids:
    article = fetch_pubmed_article(pmid)

    if article is None:
        continue

    # Skip empty or very short abstracts
    if not article["abstract"] or len(article["abstract"]) < 50:
        print(f"Skipping PMID {pmid} (abstract too short)")
        continue

    articles.append(article)

    cleaned = article.copy()
    cleaned["abstract_clean"] = clean_abstract(article["abstract"])
    cleaned_records.append(cleaned)

    print("\n" + "=" * 90)
    print("PMID:", article["pmid"])
    print("TITLE:", article["title"])
    print("YEAR:", article["year"])
    print("ABSTRACT LENGTH:", len(article["abstract"]))

    time.sleep(0.1)


# -------- Save RAW JSON --------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(articles, f, indent=2, ensure_ascii=False)

print(f"\n✅ Saved articles to {OUTPUT_FILE}")


# -------- Save CLEAN JSON --------
with open(CLEAN_FILE, "w", encoding="utf-8") as f:
    json.dump(cleaned_records, f, indent=2, ensure_ascii=False)

print(f"✅ Saved cleaned dataset to {CLEAN_FILE}")

🔬 PubMed Search Tool
----------------------------------------
Enter PubMed search query: histones

🔍 Searching PubMed...
Total PMIDs found: 50
Skipping PMID 41925818 (abstract too short)

PMID: 41925445
TITLE: When loss is gain: truncating mutations in additional sex combs (ASXL) gene family in cancer and neurodevelopment.
YEAR: 2026
ABSTRACT LENGTH: 904

PMID: 41925392
TITLE: Citrate-mediated control of chromatin and mitosis.
YEAR: 2026
ABSTRACT LENGTH: 414

PMID: 41925243
TITLE: Phytochrome-Mediated Light Perception in Dodders Drives Haustorium Development Through Epigenetic Mechanisms.
YEAR: 2026
ABSTRACT LENGTH: 1543

PMID: 41925229
TITLE: Trans-histone crosstalk establishes distinct H3K79 methylation zones with differential transcriptional functions.
YEAR: 2026
ABSTRACT LENGTH: 1556

PMID: 41924779
TITLE: Epigenetic rewiring and T cell exhaustion in HBV-induced HCC with implications for precision therapies.
YEAR: 2026
ABSTRACT LENGTH: 1880

PMID: 41924652
TITLE: HMGB1 as Double-Ed

# Chunk abstracts for RAG

In [12]:
import json
import re

CHUNK_SIZE = 400
CHUNK_OVERLAP = 80


def sentence_split(text: str):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if s.strip()]


def chunk_text(text: str, chunk_size=400, overlap=80):
    sentences = sentence_split(text)

    chunks = []
    current_chunk = []
    current_len = 0

    for sent in sentences:
        words = sent.split()
        sent_len = len(words)

        if current_len + sent_len > chunk_size:
            chunks.append(" ".join(current_chunk))

            overlap_words = " ".join(current_chunk).split()[-overlap:]
            current_chunk = overlap_words.copy()
            current_len = len(current_chunk)

        current_chunk.extend(words)
        current_len += sent_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


def chunk_records(cleaned_records):
    all_chunks = []

    for rec in cleaned_records:
        text = rec["title"] + ". " + rec.get("abstract_clean", "")

        if not text:
            continue

        chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)

        for i, chunk in enumerate(chunks):
            if len(chunk.split()) < 30:
                continue

            all_chunks.append({
                "pmid": rec["pmid"],
                "title": rec["title"],
                "journal": rec["journal"],
                "year": rec["year"],
                "chunk_id": i,
                "chunk_uid": f"{rec['pmid']}_chunk_{i}",
                "text": chunk
            })

    return all_chunks


# Save chunk metadata file

# Load cleaned records
with open("biolipids_pubmed_clean.json", "r", encoding="utf-8") as f:
    cleaned_records = json.load(f)

# Create chunks
all_chunks = chunk_records(cleaned_records)

# Save chunk metadata
with open("chunk_metadata.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print("Saved chunk_metadata.json")
print("Total chunks:", len(all_chunks))

Saved chunk_metadata.json
Total chunks: 46


In [8]:
# ==============================
# BioBERT Embeddings + FAISS
# ==============================

!pip install -q sentence-transformers faiss-cpu


# Creating embeddings from chunks

In [13]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json

print("Creating embeddings from chunks...")

# Load chunks
with open("chunk_metadata.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [c["text"] for c in chunks]

# Load embedding model
model = SentenceTransformer(
    "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)

# Create embeddings
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS index built:", index.ntotal)

# Save index
faiss.write_index(index, "biobert_faiss.index")

# Save embeddings
np.save("biobert_embeddings.npy", embeddings)

print("Saved FAISS index and embeddings")

Creating embeddings from chunks...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (46, 768)
FAISS index built: 46
Saved FAISS index and embeddings


# Retrieval

In [17]:
import os
print(os.listdir())

['.config', 'chunk_metadata.json', 'bioethanol_pubmed.json', 'biobert_faiss.index', 'histones_pubmed.json', 'HDAC_pubmed_clean.json', 'HDAC_pubmed.json', 'pubmed_rag', 'biolipids_pubmed.json', 'biobert_embeddings.npy', 'biolipids_pubmed_clean.json', 'bioethanol_pubmed_clean.json', 'histones_pubmed_clean.json', 'sample_data']


In [16]:
from sentence_transformers import SentenceTransformer
import faiss
import json

print("Loading FAISS index and metadata...")

# Load index
index = faiss.read_index("biobert_faiss.index")

# Load metadata
with open("chunk_metadata.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# Load embedding model
model = SentenceTransformer(
    "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)

print("\nLiterature RAG ready. Ask questions.\n")

while True:
    query = input("Ask question (or type exit): ")

    if query.lower() == "exit":
        break

    # Embed query
    q_emb = model.encode([query], normalize_embeddings=True)

    # Search FAISS
    D, I = index.search(q_emb, k=5)

    print("\nTop Results:\n")

    for rank, idx in enumerate(I[0]):
        chunk = chunks[idx]
        print("=" * 80)
        print(f"Rank {rank+1}")
        print("PMID:", chunk["pmid"])
        print("Title:", chunk["title"])
        print("Year:", chunk["year"])
        print("Text:", chunk["text"][:400], "...")

Loading FAISS index and metadata...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Literature RAG ready. Ask questions.

Ask question (or type exit): microbial lipid production wastewater

Top Results:

Rank 1
PMID: 38242026
Title: Bibliometric insights into microalgae cultivation in wastewater: Trends and future prospects for biolipid production and environmental sustainability.
Year: 2024
Text: Bibliometric insights into microalgae cultivation in wastewater: Trends and future prospects for biolipid production and environmental sustainability.. Cultivation of microalgae in wastewater stream has been extensively reported, especially for simultaneous production of biolipid and wastewater treatment process. This study aimed to derive the research trend and focus on biolipid production from m ...
Rank 2
PMID: 37866181
Title: Biodiesel production and properties estimation from food waste and domestic wastewater by Rhodosporidium toruloides.
Year: 2023
Text: Biodiesel production and properties estimation from food waste and domestic wastewater by Rhodosporidium toruloide

KeyboardInterrupt: Interrupted by user

In [19]:
# Merge jason files

import json
import glob

clean_files = glob.glob("*_pubmed_clean.json")

all_records = []
seen_pmids = set()

for file in clean_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

        for rec in data:
            if rec["pmid"] not in seen_pmids:
                all_records.append(rec)
                seen_pmids.add(rec["pmid"])

print("Total unique papers:", len(all_records))

with open("merged_pubmed_clean.json", "w", encoding="utf-8") as f:
    json.dump(all_records, f, indent=2, ensure_ascii=False)

print("Saved merged_pubmed_clean.json")

Total unique papers: 138
Saved merged_pubmed_clean.json
